# Seasonal Agriculture Performance Analysis

**Major Project – VOIS AICTE Batch 1 2026–2027**

This notebook analyzes the provided agricultural dataset to investigate how agricultural performance varies across Kharif, Rabi and Zaid seasons.


## 1. Problem Statement

Agricultural activities are influenced by seasonal variations in environmental conditions, farming practices, resource availability and market conditions. This project analyzes the available agricultural data to identify meaningful seasonal patterns, trends, relationships and variations in agricultural performance.


## 2. Objectives

- Explore and understand the dataset.
- Check and prepare the data for analysis.
- Compare yield and profitability across seasons.
- Analyze irrigation and crop-level differences.
- Examine relationships between environmental/resource variables and yield.
- Identify significant observations and unusual patterns.
- Use statistical summaries and visualizations.
- Develop evidence-based conclusions and recommendations.


## 3. Analytical Questions

1. How does agricultural yield vary across seasons?
2. How does average profit vary across seasons?
3. Which irrigation method is associated with the highest average yield and profit?
4. How does crop performance differ across seasons?
5. Which numerical factors have the strongest relationship with yield?
6. Are there notable differences in rainfall, temperature, water efficiency or disease/pest risk between seasons?


## 4. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 5. Load Dataset

In [ ]:
df = pd.read_csv("seasonal_agriculture_performance_dataset.csv")
print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


## 6. Initial Exploration

In [ ]:
print("Shape:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())


In [ ]:
print("Data types:")
display(df.dtypes)
print("\nDataset information:")
df.info()


## 7. Data Quality Check

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)

print("Total missing cells:", int(df.isnull().sum().sum()))
print("\nColumns with missing values:")
display(missing[missing > 0])

print("\nDuplicate rows:", int(df.duplicated().sum()))


### Data Quality Interpretation

The dataset contains **4,000 rows and 28 columns**, with **120 missing cells** and **0 duplicate rows**. Missing values are reported rather than silently replacing them. Pandas aggregation functions exclude missing observations from the relevant calculation.


## 8. Descriptive Statistics

In [ ]:
display(df.select_dtypes(include=np.number).describe().T)


## 9. Season Distribution

In [ ]:
season_counts = df["Season"].value_counts()
display(season_counts)

plt.figure(figsize=(7,4))
plt.bar(season_counts.index, season_counts.values)
plt.title("Number of Records by Season")
plt.xlabel("Season")
plt.ylabel("Number of Records")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()


## 10. Seasonal Performance Summary

In [ ]:
season_summary = df.groupby("Season").agg(
    Records=("Farm_ID", "count"),
    Average_Yield_Tonnes_Ha=("Yield_Tonnes_Ha", "mean"),
    Average_Profit_INR=("Profit_INR", "mean"),
    Average_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Average_Rainfall_mm=("Rainfall_mm", "mean"),
    Average_Temperature_C=("Avg_Temperature_C", "mean"),
    Average_Disease_Pest_Risk_pct=("Disease_Pest_Risk_pct", "mean")
).round(2)

display(season_summary)


## 11. Result 1 – Average Yield by Season

In [ ]:
season_yield = df.groupby("Season")["Yield_Tonnes_Ha"].mean().sort_values(ascending=False)

display(season_yield.round(2).to_frame("Average Yield (Tonnes/Ha)"))

plt.figure(figsize=(8,5))
plt.bar(season_yield.index, season_yield.values)
plt.title("Average Yield by Season")
plt.xlabel("Season")
plt.ylabel("Yield (Tonnes/Ha)")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Highest average yield: {season_yield.idxmax()} ({season_yield.max():.2f} tonnes/ha)")
print(f"Lowest average yield: {season_yield.idxmin()} ({season_yield.min():.2f} tonnes/ha)")


**Finding:** Kharif has the highest average yield at approximately 5.64 tonnes/ha, while Zaid has the lowest at approximately 4.67 tonnes/ha.

## 12. Result 2 – Average Profit by Season

In [ ]:
season_profit = df.groupby("Season")["Profit_INR"].mean().sort_values(ascending=False)

display(season_profit.round(2).to_frame("Average Profit (INR)"))

plt.figure(figsize=(8,5))
plt.bar(season_profit.index, season_profit.values)
plt.title("Average Profit by Season")
plt.xlabel("Season")
plt.ylabel("Average Profit (₹)")
plt.axhline(0, linewidth=0.8)
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Highest average profit: {season_profit.idxmax()} (₹{season_profit.max():,.0f})")
print(f"Lowest average profit: {season_profit.idxmin()} (₹{season_profit.min():,.0f})")


**Finding:** Kharif has the highest average profit at approximately ₹178,915. Rabi averages ₹87,689, while Zaid averages −₹24,805.

## 13. Result 3 – Irrigation Method Analysis

In [ ]:
irrigation_summary = df.groupby("Irrigation_Method").agg(
    Records=("Farm_ID", "count"),
    Average_Yield=("Yield_Tonnes_Ha", "mean"),
    Average_Profit=("Profit_INR", "mean"),
    Average_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean")
).round(2).sort_values("Average_Yield", ascending=False)

display(irrigation_summary)

plt.figure(figsize=(9,5))
plt.bar(irrigation_summary.index, irrigation_summary["Average_Yield"])
plt.title("Average Yield by Irrigation Method")
plt.xlabel("Irrigation Method")
plt.ylabel("Yield (Tonnes/Ha)")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

print("Highest average-yield irrigation method:", irrigation_summary["Average_Yield"].idxmax())
print("Highest average yield:", f'{irrigation_summary["Average_Yield"].max():.2f} tonnes/ha')


**Finding:** Drip irrigation records the highest average yield at approximately 6.62 tonnes/ha and the highest average profit at approximately ₹219,626 among the irrigation methods analyzed. This is an association, not proof of causation.

## 14. Result 4 – Crop and Seasonal Analysis

In [ ]:
crop_season = df.groupby(["Season", "Crop"]).agg(
    Records=("Farm_ID", "count"),
    Average_Yield=("Yield_Tonnes_Ha", "mean"),
    Average_Profit=("Profit_INR", "mean")
).reset_index().round(2)

display(crop_season.sort_values(["Season", "Average_Yield"], ascending=[True, False]).head(20))


In [ ]:
sugarcane = crop_season[crop_season["Crop"].astype(str).str.lower().eq("sugarcane")].sort_values("Season")
display(sugarcane)

plt.figure(figsize=(8,5))
plt.bar(sugarcane["Season"], sugarcane["Average_Yield"])
plt.title("Sugarcane Average Yield Across Seasons")
plt.xlabel("Season")
plt.ylabel("Yield (Tonnes/Ha)")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()


**Finding:** Sugarcane has a substantially higher yield scale than the other crops. Its Kharif average yield is approximately 53.46 tonnes/ha.

## 15. Result 5 – Relationships with Yield

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
yield_correlations = df[numeric_cols].corr(numeric_only=True)["Yield_Tonnes_Ha"].sort_values(ascending=False)

display(yield_correlations.to_frame("Correlation with Yield"))


**Finding:** Correlation analysis identifies numerical variables that move together with yield. Correlation does not establish causation; it highlights relationships for further investigation.

## 16. Environmental and Resource Comparison

In [ ]:
environment_summary = df.groupby("Season").agg(
    Rainfall_mm=("Rainfall_mm", "mean"),
    Temperature_C=("Avg_Temperature_C", "mean"),
    Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Disease_Pest_Risk_pct=("Disease_Pest_Risk_pct", "mean")
).round(2)

display(environment_summary)


## 17. State-Level Performance

In [ ]:
state_summary = df.groupby("State").agg(
    Records=("Farm_ID", "count"),
    Average_Yield=("Yield_Tonnes_Ha", "mean"),
    Average_Profit=("Profit_INR", "mean")
).round(2).sort_values("Average_Yield", ascending=False)

display(state_summary)


## 18. Key Findings

In [ ]:
print("KEY FINDINGS")
print("-" * 60)
print(f"1. Highest seasonal average yield: {season_yield.idxmax()} ({season_yield.max():.2f} tonnes/ha).")
print(f"2. Lowest seasonal average yield: {season_yield.idxmin()} ({season_yield.min():.2f} tonnes/ha).")
print(f"3. Highest seasonal average profit: {season_profit.idxmax()} (₹{season_profit.max():,.0f}).")
print(f"4. Lowest seasonal average profit: {season_profit.idxmin()} (₹{season_profit.min():,.0f}).")
print(f"5. Highest average-yield irrigation method: {irrigation_summary['Average_Yield'].idxmax()} ({irrigation_summary['Average_Yield'].max():.2f} tonnes/ha).")
print("6. Sugarcane has a substantially higher yield scale than the other crops.")


## 19. Conclusions

- Kharif shows the strongest average performance in terms of yield and profit.
- Rabi shows moderate performance with positive average profitability.
- Zaid has the lowest average yield and negative average profit in the available data.
- Irrigation method is associated with differences in yield and profit; drip irrigation shows the strongest average results.
- Crop type has a major influence on yield scale, illustrated by Sugarcane.
- Environmental, resource and farming variables should be considered together when interpreting seasonal performance.


## 20. Recommendations

1. Investigate factors contributing to strong Kharif performance.
2. Examine the causes of negative average profitability in Zaid.
3. Evaluate efficient irrigation practices while considering crop and location differences.
4. Plan crops and resources according to seasonal environmental conditions.
5. Monitor rainfall, temperature and disease/pest risk during seasonal planning.
6. Use crop-specific benchmarks instead of comparing all crops only by raw yield.
7. Collect additional historical data for year-to-year seasonal analysis.


## 21. Future Scope

- Machine-learning-based crop-yield prediction
- Seasonal profit forecasting
- Real-time weather and market-price integration
- Soil, fertilizer and pesticide analysis
- Interactive agricultural dashboards
- Geographical/state/district-level visualization
- Disease and pest risk early-warning systems


## 22. Final Summary

This project used Python-based data analytics to study **Seasonal Agriculture Performance**. The analysis covered data quality, seasonal performance, profitability, irrigation, crop-level patterns and numerical relationships. The findings provide evidence-based insights that can support better seasonal agricultural planning and identify areas requiring further investigation.
